# 04 - Uncertainty Modelling

In this notebook I define how uncertainty will be applied to the nominal cooling-load values before the Monte Carlo simulation is built.

Notebook 03 showed that the deterministic assessment passes in both Scenario Alpha and Scenario Bravo, but several subsystems are close enough to capacity that uncertainty could still be important. This notebook therefore prepares the uncertainty assumptions that will be used in Notebook 05.

## Purpose of this notebook

The aim of this stage is not to run the full Monte Carlo simulation yet. Instead, I define and check the load sampling assumptions.

The notebook:

* recreates the local modelling dataset from the updated workbook;
* excludes embedded propagation rows from downstream local sampling;
* creates margin cases A, B and C;
* defines triangular and truncated normal distribution parameters;
* uses a 95% interval assumption for the truncated normal standard deviation;
* checks that sampled values are non-negative and remain inside the defined uncertainty bounds.

## Import libraries and set file paths

I keep the code inside this notebook so that the modelling assumptions are visible and reproducible.

In [ ]:
# Importing the required libraries
from pathlib import Path
import pandas as pd
import numpy as np

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    plotting_available = True
except ModuleNotFoundError:
    plotting_available = False
    print("matplotlib and seaborn are not installed yet, so plots will be skipped.")

try:
    from scipy.stats import truncnorm
    scipy_available = True
except ModuleNotFoundError:
    scipy_available = False
    print("scipy is not installed yet, so truncated normal sampling checks will be skipped.")

# Find the project root. This works if the notebook is run from the repo root or from the notebooks folder.
current_path = Path.cwd().resolve()
if (current_path / "data" / "raw" / "data.xlsx").exists():
    project_root = current_path
elif (current_path.parent / "data" / "raw" / "data.xlsx").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError("Could not find data/raw/data.xlsx. Please open VS Code at the repository folder.")

raw_data_path = project_root / "data" / "raw" / "data.xlsx"
tables_output_path = project_root / "outputs" / "tables"
figures_output_path = project_root / "outputs" / "figures"

tables_output_path.mkdir(parents=True, exist_ok=True)
figures_output_path.mkdir(parents=True, exist_ok=True)

random_seed = 19046600
rng = np.random.default_rng(random_seed)

print(f"Project root: {project_root}")
print(f"Raw data path: {raw_data_path}")
print(f"Random seed: {random_seed}")

## Load the workbook

I load the updated workbook directly. This keeps the uncertainty modelling aligned with the EDA and deterministic baseline notebooks.

In [ ]:
# Load the workbook
excel_file = pd.ExcelFile(raw_data_path)
sheet_names = excel_file.sheet_names
raw_data = pd.read_excel(raw_data_path, sheet_name=sheet_names[0])

print("Sheet names in workbook:")
print(sheet_names)
print(f"Raw dataset shape: {raw_data.shape[0]} rows and {raw_data.shape[1]} columns")
display(raw_data.head())

## Recreate the local modelling dataset

I reshape the workbook into one row per source row and subsystem. This is similar to Notebook 02, but here I keep only the fields needed for uncertainty modelling.

The embedded propagation rows are flagged so they can be excluded from local downstream sampling:

* `ITEM_000955` is excluded from local CW sampling;
* `ITEM_001103` is excluded from local FW sampling.

These upstream loads will be replaced later by sampled upstream totals in the Monte Carlo cascade.

In [ ]:
systems = ["HVAC", "CW", "FW"]
records = []

for row_number, row in raw_data.iterrows():
    for system in systems:
        system_columns = {
            "alpha_load_kw": f"{system} Scenario Alpha Load",
            "bravo_load_kw": f"{system} Scenario Bravo Load",
            "cmm": f"{system} CMM",
            "cum": f"{system} CUM",
            "dbm": f"{system} DBM",
            "igm": f"{system} IGM"
        }

        if all(pd.isna(row[column]) for column in system_columns.values()):
            continue

        record = {
            "source_row": row_number,
            "record_id": row_number + 1,
            "pbs_code": row["PBS Code"],
            "teamcenter_id": row["Teamcenter ID"],
            "bom_id": row["BOM ID"],
            "item_description": row["Item Description"],
            "system": system
        }

        for new_column, original_column in system_columns.items():
            record[new_column] = pd.to_numeric(row[original_column], errors="coerce")

        for scenario in ["alpha", "bravo"]:
            load_column = f"{scenario}_load_kw"
            record[f"{scenario}_load_was_blank"] = pd.isna(record[load_column])
            record[f"{scenario}_load_model_kw"] = 0 if pd.isna(record[load_column]) else record[load_column]

        for margin_column in ["cmm", "cum", "dbm", "igm"]:
            record[f"{margin_column}_was_blank"] = pd.isna(record[margin_column])
            record[f"{margin_column}_model"] = 0 if pd.isna(record[margin_column]) else record[margin_column]

        record["is_propagation_row"] = (
            (system == "CW" and row["Item Description"] == "ITEM_000955") or
            (system == "FW" and row["Item Description"] == "ITEM_001103")
        )

        if system == "CW" and row["Item Description"] == "ITEM_000955":
            record["propagation_type"] = "hvac_to_cw"
        elif system == "FW" and row["Item Description"] == "ITEM_001103":
            record["propagation_type"] = "cw_to_fw"
        else:
            record["propagation_type"] = "local"

        record["margin_case_a"] = record["cum_model"]
        record["margin_case_b"] = record["cmm_model"] + record["cum_model"]
        record["margin_case_c"] = (
            record["cmm_model"] + record["cum_model"] + record["dbm_model"] + record["igm_model"]
        )

        records.append(record)

modelling_records = pd.DataFrame(records)
local_records = modelling_records[~modelling_records["is_propagation_row"]].copy()

print(f"All subsystem records: {len(modelling_records)}")
print(f"Local records used for uncertainty modelling: {len(local_records)}")
display(local_records.head())

## Check excluded propagation rows

I check the excluded rows explicitly because this is the main step that prevents the Monte Carlo model from double-counting cascaded loads.

In [ ]:
excluded_propagation_rows = modelling_records[modelling_records["is_propagation_row"]].copy()

excluded_columns = [
    "source_row", "record_id", "item_description", "system", "propagation_type",
    "alpha_load_model_kw", "bravo_load_model_kw"
]

excluded_propagation_rows = excluded_propagation_rows[excluded_columns]
excluded_propagation_rows.to_csv(tables_output_path / "uncertainty_excluded_propagation_rows.csv", index=False)
display(excluded_propagation_rows)

## Margin cases

I use the three margin cases agreed for the project:

* Case A = `CUM`
* Case B = `CMM + CUM`
* Case C = `CMM + CUM + DBM + IGM`

These are treated as uncertainty assumptions rather than as a single correct interpretation of the engineering margins.

In [ ]:
margin_case_summary = (
    local_records
    .groupby("system")
    .agg(
        records=("record_id", "count"),
        case_a_mean=("margin_case_a", "mean"),
        case_a_max=("margin_case_a", "max"),
        case_b_mean=("margin_case_b", "mean"),
        case_b_max=("margin_case_b", "max"),
        case_c_mean=("margin_case_c", "mean"),
        case_c_max=("margin_case_c", "max")
    )
    .reset_index()
)

margin_case_summary.to_csv(tables_output_path / "uncertainty_margin_case_summary.csv", index=False)
display(margin_case_summary)

## Define distribution parameter tables

For each local load record, scenario and margin case, I define parameters for two distributions.

**Triangular distribution**

```text
lower = max(0, nominal * (1 - uncertainty_pct))
mode  = nominal
upper = nominal * (1 + uncertainty_pct)
```

**Truncated normal distribution**

```text
mean  = nominal
lower = max(0, nominal * (1 - uncertainty_pct))
upper = nominal * (1 + uncertainty_pct)
sigma = (uncertainty_pct * nominal) / 1.96
```

The `1.96` value means I am treating the uncertainty band as an approximate 95% interval around the nominal load before truncation. If the nominal load or uncertainty percentage is zero, the sampled value should remain equal to the nominal load.

In [ ]:
parameter_rows = []

margin_cases = {
    "Case A - CUM": "margin_case_a",
    "Case B - CMM + CUM": "margin_case_b",
    "Case C - CMM + CUM + DBM + IGM": "margin_case_c"
}

scenarios = {
    "Scenario Alpha": "alpha_load_model_kw",
    "Scenario Bravo": "bravo_load_model_kw"
}

for _, row in local_records.iterrows():
    for scenario_name, load_column in scenarios.items():
        nominal_load = float(row[load_column])

        for margin_case_name, margin_column in margin_cases.items():
            uncertainty_pct = float(row[margin_column])
            lower = max(0, nominal_load * (1 - uncertainty_pct))
            upper = nominal_load * (1 + uncertainty_pct)

            if nominal_load == 0 or uncertainty_pct == 0:
                sigma = 0
            else:
                sigma = (uncertainty_pct * nominal_load) / 1.96

            parameter_rows.append({
                "record_id": row["record_id"],
                "source_row": row["source_row"],
                "pbs_code": row["pbs_code"],
                "teamcenter_id": row["teamcenter_id"],
                "bom_id": row["bom_id"],
                "item_description": row["item_description"],
                "system": row["system"],
                "scenario": scenario_name,
                "margin_case": margin_case_name,
                "nominal_load_kw": nominal_load,
                "uncertainty_pct": uncertainty_pct,
                "triangular_lower_kw": lower,
                "triangular_mode_kw": nominal_load,
                "triangular_upper_kw": upper,
                "truncnorm_lower_kw": lower,
                "truncnorm_mean_kw": nominal_load,
                "truncnorm_upper_kw": upper,
                "truncnorm_sigma_kw": sigma,
                "is_degenerate": nominal_load == 0 or uncertainty_pct == 0
            })

distribution_parameters = pd.DataFrame(parameter_rows)
distribution_parameters.to_csv(tables_output_path / "uncertainty_distribution_parameters.csv", index=False)

print(f"Distribution parameter rows: {len(distribution_parameters)}")
display(distribution_parameters.head())

## Parameter summary

I summarise the parameter table to check that the uncertainty cases behave as expected. Case C should generally create the widest uncertainty bands.

In [ ]:
parameter_summary = (
    distribution_parameters
    .groupby(["system", "scenario", "margin_case"])
    .agg(
        records=("record_id", "count"),
        positive_nominal_records=("nominal_load_kw", lambda x: (x > 0).sum()),
        mean_nominal_load_kw=("nominal_load_kw", "mean"),
        max_nominal_load_kw=("nominal_load_kw", "max"),
        mean_uncertainty_pct=("uncertainty_pct", "mean"),
        max_uncertainty_pct=("uncertainty_pct", "max"),
        mean_sigma_kw=("truncnorm_sigma_kw", "mean"),
        max_upper_kw=("triangular_upper_kw", "max")
    )
    .reset_index()
)

parameter_summary.to_csv(tables_output_path / "uncertainty_parameter_summary.csv", index=False)
display(parameter_summary)

## Sampling functions for validation

I define simple sampling functions so that I can validate the assumptions before running the full Monte Carlo simulation in Notebook 05.

In [ ]:
def sample_triangular(row, sample_size, rng):
    nominal = row["nominal_load_kw"]
    lower = row["triangular_lower_kw"]
    upper = row["triangular_upper_kw"]

    if row["is_degenerate"]:
        return np.full(sample_size, nominal)

    return rng.triangular(lower, nominal, upper, sample_size)


def sample_truncated_normal(row, sample_size, rng):
    nominal = row["nominal_load_kw"]
    lower = row["truncnorm_lower_kw"]
    upper = row["truncnorm_upper_kw"]
    sigma = row["truncnorm_sigma_kw"]

    if row["is_degenerate"]:
        return np.full(sample_size, nominal)

    if not scipy_available:
        raise ModuleNotFoundError("scipy is required for truncated normal sampling.")

    a = (lower - nominal) / sigma
    b = (upper - nominal) / sigma
    return truncnorm.rvs(a, b, loc=nominal, scale=sigma, size=sample_size, random_state=rng)

## Validate example sampling

I select one positive-load record for each system and scenario. I use the largest non-zero uncertain load within each group, so this remains a simple validation check rather than the final Monte Carlo simulation.


In [ ]:
positive_parameters = distribution_parameters[
    (distribution_parameters["nominal_load_kw"] > 0) &
    (distribution_parameters["uncertainty_pct"] > 0)
].copy()

system_order = ["HVAC", "CW", "FW"]
scenario_order = ["Scenario Alpha", "Scenario Bravo"]

example_parameters = (
    positive_parameters
    .sort_values("nominal_load_kw", ascending=False)
    .groupby(["system", "scenario"])
    .head(1)
    .assign(
        system=lambda frame: pd.Categorical(frame["system"], categories=system_order, ordered=True),
        scenario=lambda frame: pd.Categorical(frame["scenario"], categories=scenario_order, ordered=True)
    )
    .sort_values(["scenario", "system"])
    .reset_index(drop=True)
)

example_parameters.to_csv(tables_output_path / "uncertainty_example_parameter_rows.csv", index=False)
display(example_parameters)


In [ ]:
validation_rows = []
sample_size = 5000

for _, row in example_parameters.iterrows():
    triangular_samples = sample_triangular(row, sample_size, rng)

    validation_rows.append({
        "distribution": "Triangular",
        "system": row["system"],
        "scenario": row["scenario"],
        "margin_case": row["margin_case"],
        "record_id": row["record_id"],
        "nominal_load_kw": row["nominal_load_kw"],
        "lower_kw": row["triangular_lower_kw"],
        "upper_kw": row["triangular_upper_kw"],
        "sample_mean_kw": triangular_samples.mean(),
        "sample_min_kw": triangular_samples.min(),
        "sample_max_kw": triangular_samples.max(),
        "non_negative_check": bool((triangular_samples >= 0).all()),
        "within_bounds_check": bool(
            (triangular_samples >= row["triangular_lower_kw"]).all() and
            (triangular_samples <= row["triangular_upper_kw"]).all()
        )
    })

    if scipy_available:
        truncnorm_samples = sample_truncated_normal(row, sample_size, rng)

        validation_rows.append({
            "distribution": "Truncated normal",
            "system": row["system"],
            "scenario": row["scenario"],
            "margin_case": row["margin_case"],
            "record_id": row["record_id"],
            "nominal_load_kw": row["nominal_load_kw"],
            "lower_kw": row["truncnorm_lower_kw"],
            "upper_kw": row["truncnorm_upper_kw"],
            "sample_mean_kw": truncnorm_samples.mean(),
            "sample_min_kw": truncnorm_samples.min(),
            "sample_max_kw": truncnorm_samples.max(),
            "non_negative_check": bool((truncnorm_samples >= 0).all()),
            "within_bounds_check": bool(
                (truncnorm_samples >= row["truncnorm_lower_kw"]).all() and
                (truncnorm_samples <= row["truncnorm_upper_kw"]).all()
            )
        })

sampling_validation = pd.DataFrame(validation_rows)
sampling_validation.to_csv(tables_output_path / "uncertainty_sampling_validation.csv", index=False)
display(sampling_validation)

## Zero-load and zero-uncertainty checks

Records with zero nominal load or zero uncertainty should not vary when sampled. This is important because blank scenario fields were converted to zero for modelling.

In [ ]:
degenerate_parameters = distribution_parameters[distribution_parameters["is_degenerate"]].copy()

degenerate_check_rows = []
for _, row in degenerate_parameters.head(20).iterrows():
    triangular_samples = sample_triangular(row, 100, rng)

    degenerate_check_rows.append({
        "distribution": "Triangular",
        "record_id": row["record_id"],
        "scenario": row["scenario"],
        "margin_case": row["margin_case"],
        "nominal_load_kw": row["nominal_load_kw"],
        "uncertainty_pct": row["uncertainty_pct"],
        "all_samples_equal_nominal": bool(np.allclose(triangular_samples, row["nominal_load_kw"]))
    })

    if scipy_available:
        truncnorm_samples = sample_truncated_normal(row, 100, rng)

        degenerate_check_rows.append({
            "distribution": "Truncated normal",
            "record_id": row["record_id"],
            "scenario": row["scenario"],
            "margin_case": row["margin_case"],
            "nominal_load_kw": row["nominal_load_kw"],
            "uncertainty_pct": row["uncertainty_pct"],
            "all_samples_equal_nominal": bool(np.allclose(truncnorm_samples, row["nominal_load_kw"]))
        })

degenerate_sampling_check = pd.DataFrame(degenerate_check_rows)
degenerate_sampling_check.to_csv(tables_output_path / "uncertainty_degenerate_sampling_check.csv", index=False)
display(degenerate_sampling_check)

## Example distribution plots

These plots are used to visually check that the distributions behave sensibly. I include one representative positive-load record for each subsystem in each scenario so that HVAC, CW and FW are all visible before the full Monte Carlo simulation is run in Notebook 05.


In [ ]:
if plotting_available:
    plot_rows = []
    plot_sample_size = 3000

    for _, row in example_parameters.iterrows():
        triangular_samples = sample_triangular(row, plot_sample_size, rng)
        for value in triangular_samples:
            plot_rows.append({
                "distribution": "Triangular",
                "system": row["system"],
                "scenario": row["scenario"],
                "margin_case": row["margin_case"],
                "sample_load_kw": value
            })

        if scipy_available:
            truncnorm_samples = sample_truncated_normal(row, plot_sample_size, rng)
            for value in truncnorm_samples:
                plot_rows.append({
                    "distribution": "Truncated normal",
                    "system": row["system"],
                    "scenario": row["scenario"],
                    "margin_case": row["margin_case"],
                    "sample_load_kw": value
                })

    plot_samples = pd.DataFrame(plot_rows)

    graph = sns.displot(
        data=plot_samples,
        x="sample_load_kw",
        hue="distribution",
        col="system",
        row="scenario",
        col_order=system_order,
        row_order=scenario_order,
        kind="hist",
        bins=35,
        common_bins=False,
        facet_kws={"sharex": False, "sharey": False},
        alpha=0.55
    )
    graph.fig.suptitle("Representative Load Sampling Distributions", y=1.02)
    graph.set_axis_labels("Sampled load (kW)", "Count")
    graph.savefig(figures_output_path / "uncertainty_example_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(9, 5))
    sns.barplot(
        data=parameter_summary,
        x="system",
        y="mean_uncertainty_pct",
        hue="margin_case"
    )
    plt.title("Mean Uncertainty Percentage by System and Margin Case")
    plt.xlabel("System")
    plt.ylabel("Mean uncertainty percentage")
    plt.tight_layout()
    plt.savefig(figures_output_path / "uncertainty_margin_case_summary.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping plots because matplotlib and seaborn are not installed in this Python environment.")


## Uncertainty modelling decisions

The uncertainty modelling assumptions are now ready to carry into Notebook 05.

The key decisions are:

1. Only local load rows are sampled.
2. Embedded propagation rows are excluded from local downstream sampling.
3. Blank scenario loads are converted to zero and remain zero when sampled.
4. Three margin cases are used to test sensitivity to margin interpretation.
5. The triangular distribution uses the nominal load as the mode.
6. The truncated normal distribution uses the nominal load as the mean and applies the 95% interval assumption for sigma.

This stage does not estimate overload probability yet. The next notebook should use these parameter tables to run the Monte Carlo simulation and propagate sampled loads through HVAC, CW and FW.